In [2]:
# Importing libraries and Setup
from dotenv import load_dotenv
from openai import OpenAI
import json 
import os
import requests
from pypdf import PdfReader
import gradio as gr
from IPython.display import Markdown, display

In [3]:
# Load the API keys
load_dotenv(override=True)

# Instance of openai
openai = OpenAI()

In [4]:
# Pushover configuration
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


# Function to send notification to pushover app
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, 'token': pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


In [51]:
# Testing
push("Hey!")

Push: Hey!


In [5]:
def record_user_details(email, name='Name not provided', notes='not provided'):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"


def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [6]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional into about the conversation that's worth recording to give context"}
        }
    },
    "required": ["email"],
    "additionalProperties": False
}

record_unknown_questions_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }

}

In [7]:
# Adding tools
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_questions_json}
]

tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional into about the conversation that's worth recording to give context"}}},
   'required': ['email'],
   'additionalProperties': False}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question'],
  

In [65]:
# This function can take a list of tool calls, and run them. This is the IF statement!!! Manually

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results 

In [66]:
# Using Python built-in globals()
# Python has a dictionary that gives us access to all global functions
globals()['record_unknown_question']("This is a really a hard question")

Push: Recording This is a really a hard question asked that I couldn't answer


'OK'

In [8]:
# This gives us a more elegant way that avoids the IF statement
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    
    return results

In [9]:
# Reading the linkedin profile
reader = PdfReader('twin/linkedin_profile.pdf')
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin+=text

# Reading resume
reader = PdfReader("twin/resume.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume+=text

# Reading summary
with open("twin/summary.txt", 'r', encoding='utf-8') as f:
    summary = f.read()

In [13]:
system_prompt = f"""
# Your role
You are an AI digital twin representing the person whose website you are on.
You interact with visitors and answer questions about their career, background,
skills, experience, projects, and professional interests.

You represent the person professionally and conversationally.

# About the person
{summary}

# LinkedIn Profile
{linkedin}

# Resume
{resume}

# Rules
Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.

- **Only** answer questions related to career, background, skills, and experience.
- **If the user asks about something unrelated or out-of-bounds, you MUST use your `record_unknown_question` tool to record it, and then steer the conversation back to professional topics.**
- If you don't know the answer to a professional question, use your `record_unknown_question` tool, and then tell the user you don't know. Never make up an answer.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.
"""

In [18]:
# chat function
def chat(message, history):
    messages = [{'role': "system", "content": system_prompt}] + history + [{'role': "user", 'content': message}]
    response = openai.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=tools)

    while response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unknown_question
Push: Recording Who is your favorite musician? asked that I couldn't answer
Tool called: record_user_details
Push: Recording interest from Name not provided with email second_test@test.com and notes not provided
